# 付録 A 回帰分析への橋渡し: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/economics-programming/appendix-a-bridge-to-regression.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
true_intercept = 100  # 真の需要曲線 Q = a - bP の切片 a
true_slope = 2        # 真の傾き b
sigma = 10            # 観測値のばらつき（標準偏差）
per_price = 50        # 価格ごとの観測値の個数
seed = 0              # 乱数の種

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## シミュレーションで作ったデータから、需要曲線を推定する

In [ ]:
import random

from lib.appendix_a.regression import fit_line, r_squared
from lib.chapter08.simulation import simulate_demand

rng = random.Random(seed)
prices: list[float] = []
quantities: list[float] = []
for price in range(5, 21):
    observations = simulate_demand(
        price, true_intercept, true_slope, sigma, per_price, rng
    )
    prices.extend([float(price)] * len(observations))
    quantities.extend(observations)
intercept, slope = fit_line(prices, quantities)
print(f"観測値の数: {len(prices)}")
print(f"推定: 切片 {intercept:.3f}、傾き {slope:.3f}"
      f"（真の値 {true_intercept}、{-true_slope}）")
print(f"決定係数 R²: {r_squared(prices, quantities):.4f}")

## 試してみよう

- `sigma` を 0 にすると、推定は真の値そのもの（切片 100、傾き -2）で、決定係数は
  1 になります。5 では R² が 0.7751、10 では 0.4561、30 では 0.0768 と、ばらつきが
  大きいほど、決定係数は小さくなります
- `sigma` が 30 でも、傾きの推定値は -1.839 で、真の値 -2 の近くにあります
  （決定係数が小さくても、傾きは、おおよそ当たります）
- `seed` を 1 にする（`sigma` は 10）と、推定は、切片 101.430、傾き -2.107 に
  変わります（乱数が違えば、推定も少しずれます）